In [2]:
# pip install PyYAML

In [4]:
import os
import json
import shutil
from pathlib import Path
import yaml
import random
from tqdm import tqdm

# 1. 경로 설정
origin_root = Path("origin_sample_dataset") # <- 여기에 원본데이터셋 경로설정
yolo_root = Path("YOLOv11Dataset") # <- 여기에 Yolo v11 변환데이터셋 경로설정
yolo_root.mkdir(parents=True, exist_ok=True)

# 2. YOLO 디렉토리 구조 생성 (images와 labels 모두 생성)
dirs = [
    "images/train",
    "images/val", 
    "images/test",
    "labels/train",
    "labels/val",
    "labels/test"
]

for d in dirs:
    (yolo_root / d).mkdir(parents=True, exist_ok=True)

# 3. 클래스 매핑 정보 생성
class_mapping = {
    ("01", 0): 0, # '01' (배) + disease 0 = 0: '배 정상'
    ("01", 1): 1, # '01' (배) + disease 1 = 1: '배검은별무늬병'
    ("01", 2): 2, # '01' (배) + disease 2 = 2: '배과수화상병'

    ("02", 0): 8, # '02' (사과) + disease 0 = 8: '사과 정상'
    ("02", 3): 3, # '02' (사과) + disease 1 = 3: '사과갈색무늬병'
    ("02", 4): 4, # '02' (사과) + disease 2 = 4: '사과과수화상병'
    ("02", 5): 5, # '02' (사과) + disease 3 = 5: '사과부란병'
    ("02", 6): 6, # '02' (사과) + disease 4 = 6: '사과점무늬낙엽병'
    ("02", 7): 7  # '02' (사과) + disease 5 = 7: '사과탄저병'
}

# JSON → YOLO 레이블 변환 함수
def convert_label(json_path, img_filename_stem, img_width, img_height):
    """
    JSON 파일을 읽어 YOLO 형식의 텍스트 라벨을 반환합니다.
    Args:
        json_path (Path): JSON 파일의 전체 경로.
        img_filename_stem (str): 이미지 파일명 (확장자 제외), JSON 파일명과 동일하다고 가정.
        img_width (int): 원본 이미지의 너비.
        img_height (int): 원본 이미지의 높이.
    Returns:
        str: YOLO 형식의 라벨 문자열 (각 라인마다 하나의 객체), 또는 변환 실패 시 빈 문자열.
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 1. 파일명에서 작물 정보 추출
    # 예: V006_80_0_00_01_01_25_0_b06_20201005_0002_S01_1.jpg.json -> '01' (배)
    # _ 기준으로 5번째 위치 (인덱스 4)
    filename_parts = img_filename_stem.split('_')
    if len(filename_parts) > 4:
        crop_code = filename_parts[4] # '01' (배) 또는 '02' (사과)
    else:
        # print(f"    ! Warning: Could not extract crop code from filename: {img_filename_stem}. Skipping.")
        return "" # 유효하지 않은 파일명이면 빈 문자열 반환

    # 2. JSON에서 질병 정보 추출
    # 'annotations' -> 'disease' 필드 값 (정수형)
    json_disease_value = data['annotations']['disease']

    # 3. class_mapping을 사용하여 최종 class_id 결정
    class_key = (crop_code, json_disease_value)
    class_id = class_mapping.get(class_key) # 매핑에 없으면 None 반환
    
    if class_id is None:
        return "" # 매핑 정보가 없으면 빈 문자열 반환

    # 바운딩 박스 처리
    bbox_lines = []
    for point in data['annotations']['points']:
        xtl = point['xtl']
        ytl = point['ytl']
        xbr = point['xbr']
        ybr = point['ybr']
        
        # YOLO 형식으로 정규화 (x_center y_center width height)
        x_center = ((xtl + xbr) / 2) / img_width
        y_center = ((ytl + ybr) / 2) / img_height
        width = (xbr - xtl) / img_width
        height = (ybr - ytl) / img_height
        
        bbox_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
    
    return "\n".join(bbox_lines)

# 개별 JSON 파일 처리 함수
def process_json(json_file: Path, target_type: str):
    """
    단일 JSON 파일을 읽어 YOLO .txt 라벨 파일을 생성합니다.
    Args:
        json_file (Path): 처리할 JSON 파일의 Path 객체.
        target_type (str): 'train', 'val', 'test' 중 하나.
    """
    img_filename_stem = json_file.stem # JSON 파일명 (확장자 제외)

    # JSON에서 이미지 크기 추출 (description 필드 사용)
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
        
        img_width = json_data['description']['width']
        img_height = json_data['description']['height']
    except KeyError as e:
        # print(f"    ! Error: Missing key {e} in JSON file {json_file}. Skipping.")
        return
    except Exception as e:
        # print(f"    ! Error reading JSON for {json_file}: {e}. Skipping.")
        return

    # 레이블 변환
    yolo_label = convert_label(json_file, img_filename_stem, img_width, img_height)
    
    # 변환된 라벨이 없으면 (예: 매핑 실패) 파일 생성 건너뛰기
    if not yolo_label:
        return

    # YOLO 형식 라벨 저장
    label_dir = yolo_root / "labels" / target_type
    txt_path = label_dir / f"{img_filename_stem}.txt" # JSON과 동일한 파일명으로 .txt 생성
    
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(yolo_label)

# 이미지 파일 복사 함수
def copy_image(src_folder: Path, img_filename: str, target_type: str):
    """
    원본 폴더에서 이미지 파일을 찾아 YOLO 이미지 폴더로 복사합니다.
    Args:
        src_folder (Path): 원본 이미지가 있는 폴더 경로
        img_filename (str): 이미지 파일명 (확장자 포함)
        target_type (str): 'train', 'val', 'test' 중 하나
    """
    # 원본 이미지 경로 찾기 (해당 폴더 내의 [원천] 폴더들에서 검색)
    src_image_path = None
    parent_folder = src_folder.parent  # Training 또는 Validation 폴더
    
    for folder in parent_folder.iterdir():
        if folder.is_dir() and folder.name.startswith("[원천]"):
            potential_image_path = folder / img_filename
            if potential_image_path.exists():
                src_image_path = potential_image_path
                break
    
    if src_image_path is None:
        # print(f"    ! Warning: Image file {img_filename} not found in any [원천] folder. Skipping.")
        return
    
    # 대상 이미지 경로
    dst_image_path = yolo_root / "images" / target_type / img_filename
    
    # 이미지 복사
    try:
        shutil.copy2(src_image_path, dst_image_path)
    except Exception as e:
        print(f"    ! Error copying image {img_filename}: {e}")

# 데이터셋 처리 함수 (이미지 복사 기능 추가)
def process_dataset(src_type: str):
    """
    원본 데이터셋의 Training/Validation 폴더를 탐색하여 JSON 파일을 처리하고
    YOLO 형식의 .txt 라벨 파일을 생성하며, 해당 이미지 파일을 복사합니다.
    Args:
        src_type (str): 'Training' 또는 'Validation'.
    """
    print(f"\nProcessing {src_type} data...")
    
    for folder in (origin_root / src_type).iterdir():
        # '[라벨]'로 시작하는 폴더만 처리
        if folder.name.startswith("[라벨]"):
            json_files = list(folder.glob("*.json"))
            random.shuffle(json_files) # 파일 목록 섞기

            if src_type == "Training":
                target_type = "train"
                print(f"  Converting JSONs from '{folder.name}' to '{target_type}' labels and copying images...")
                for json_file in tqdm(json_files, desc=f"  {target_type.upper()} processing"):
                    # 라벨 처리
                    process_json(json_file, target_type)
                    # 이미지 복사 (JSON 파일명에서 확장자 제거 후 이미지 확장자 추가)
                    img_filename = json_file.stem  # .json 제거
                    copy_image(folder, img_filename, target_type)
                    
            elif src_type == "Validation":
                # Validation 데이터는 val과 test로 5:5 분할
                split_idx = len(json_files) // 2
                val_files = json_files[:split_idx]
                test_files = json_files[split_idx:]
                
                print(f"  Converting JSONs from '{folder.name}' to 'val' labels and copying images...")
                for json_file in tqdm(val_files, desc="  VAL processing"):
                    # 라벨 처리
                    process_json(json_file, "val")
                    # 이미지 복사
                    img_filename = json_file.stem
                    copy_image(folder, img_filename, "val")
                
                print(f"  Converting JSONs from '{folder.name}' to 'test' labels and copying images...")
                for json_file in tqdm(test_files, desc="  TEST processing"):
                    # 라벨 처리
                    process_json(json_file, "test")
                    # 이미지 복사
                    img_filename = json_file.stem
                    copy_image(folder, img_filename, "test")

# 6. 데이터 처리 실행
process_dataset("Training")
process_dataset("Validation")

# 7. dataset.yaml 파일 생성
yaml_content = {
    'path': str(yolo_root.resolve()),
    'train': 'images/train',  # 실제 이미지 경로로 수정
    'val': 'images/val',
    'test': 'images/test',

    'names': { # 최종 9개 클래스 정의
        0: '배 정상',
        1: '배검은별무늬병',
        2: '배과수화상병',
        3: '사과갈색무늬병',
        4: '사과과수화상병',
        5: '사과부란병',
        6: '사과점무늬낙엽병',
        7: '사과탄저병',
        8: '사과 정상'
    }
}

with open(yolo_root / "dataset.yaml", 'w', encoding='utf-8') as f:
    yaml.dump(yaml_content, f, allow_unicode=True, sort_keys=False)

# 8. 분할 결과 통계 출력
def print_stats():
    print("\n" + "="*50)
    print("Dataset Split Statistics:")
    total_labels = 0
    total_images = 0
    
    for split in ['train', 'val', 'test']:
        label_count = len(list((yolo_root / "labels" / split).glob("*.txt")))
        image_count = len(list((yolo_root / "images" / split).glob("*")))
        total_labels += label_count
        total_images += image_count
        print(f"  - {split}: {label_count} labels, {image_count} images")
        
        # 라벨과 이미지 수가 일치하는지 확인
        if label_count != image_count:
            print(f"    ⚠️  WARNING: Label count ({label_count}) != Image count ({image_count})")
    
    print(f"\nTotal: {total_labels} labels, {total_images} images")
    print("="*50 + "\n")

print("\nJSON to YOLO conversion and image copying completed successfully!")
print(f"YOLO dataset is generated in: {yolo_root}")
print(f"  - Images: {yolo_root / 'images'}")
print(f"  - Labels: {yolo_root / 'labels'}")
print(f"  - Config: {yolo_root / 'dataset.yaml'}")
print_stats()

print("✅ Dataset is ready for YOLOv11 training!")
print("   Use the generated 'dataset.yaml' file to train your YOLOv11 model.")


Processing Training data...
  Converting JSONs from '[라벨]배_0.정상' to 'train' labels and copying images...


  TRAIN processing: 100%|██████████| 20434/20434 [01:18<00:00, 260.86it/s]


  Converting JSONs from '[라벨]배_1.질병' to 'train' labels and copying images...


  TRAIN processing: 100%|██████████| 2557/2557 [00:23<00:00, 106.93it/s]


  Converting JSONs from '[라벨]사과_0.정상' to 'train' labels and copying images...


...

  Converting JSONs from '[라벨]사과_1.질병' to 'val' labels and copying images...


  VAL processing: 100%|██████████| 518/518 [00:16<00:00, 30.73it/s]


  Converting JSONs from '[라벨]사과_1.질병' to 'test' labels and copying images...


  TEST processing: 100%|██████████| 518/518 [00:17<00:00, 30.20it/s]



JSON to YOLO conversion and image copying completed successfully!
YOLO dataset is generated in: YOLOv11Dataset
  - Images: YOLOv11Dataset\images
  - Labels: YOLOv11Dataset\labels
  - Config: YOLOv11Dataset\dataset.yaml

Dataset Split Statistics:
  - train: 59988 labels, 0 images
    ⚠️  WARNING: Label count (59988) != Image count (0)
  - val: 3754 labels, 3755 images
    ⚠️  WARNING: Label count (3754) != Image count (3755)
  - test: 3753 labels, 3756 images
    ⚠️  WARNING: Label count (3753) != Image count (3756)

Total: 67495 labels, 7511 images

✅ Dataset is ready for YOLOv11 training!
   Use the generated 'dataset.yaml' file to train your YOLOv11 model.
